# ESM Probe PFP Metrics

This notebook adds a separate PFP-style metric layer to the ESM GO explorer.
It keeps the existing matrix, kNN, linear-probe, and S2F classification summaries unchanged, and exports a new data product using the same metric families used in the S2F competitor analysis:

- AUROC, stored as `AUC` by the existing metric implementation
- AUPR
- Fmax, stored as `F_max`
- Smin, stored as `smin`

The selected-term analysis evaluates scored `(protein_id, GO term, score)` pairs over the same 60 proteins and three direct GO groups shown in the explorer. For this constrained 60-by-3 matrix, `Smin` uses information content estimated inside the selected-term evaluation matrix; the full benchmark context keeps the original GOA/ontology `Smin` from `notebooks/competitors_performance.ipynb`.

In [1]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import configparser
import json
import math
import os
import re
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

if not hasattr(np, "float"):
    np.float = float  # type: ignore[attr-defined]
if not hasattr(np, "int"):
    np.int = int  # type: ignore[attr-defined]
if not hasattr(np, "bool"):
    np.bool = bool  # type: ignore[attr-defined]

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "plm.py").exists():
    raise FileNotFoundError(f"Run from the S2F repository or notebooks directory: {PROJECT_ROOT}")
sys.path.insert(0, str(PROJECT_ROOT))

import plm
from Measures.measures import HX_py

CONFIG_PATH = PROJECT_ROOT / "s2f.conf"
config = configparser.ConfigParser()
config.read(CONFIG_PATH)

DATA_DIR = PROJECT_ROOT / "notebooks" / "esm_go_explorer" / "data"
INDEX_PATH = DATA_DIR / "index.json"
PROBE_RESULTS_PATH = DATA_DIR / "prediction_probe_results.json"
PFP_JSON_PATH = DATA_DIR / "probe_pfp_metrics.json"
PFP_METRICS_CSV_PATH = DATA_DIR / "probe_pfp_metrics.csv"
PFP_PAIR_SCORES_CSV_PATH = DATA_DIR / "probe_pfp_pair_scores.csv"
PFP_COMPETITOR_CONTEXT_CSV_PATH = DATA_DIR / "competitor_context_metrics.csv"
PFP_SPREADSHEET_JSON_PATH = DATA_DIR / "pfp_spreadsheet.json"
PFP_SPREADSHEET_CSV_PATH = DATA_DIR / "pfp_spreadsheet.csv"

CACHE_BY_ORGANISM = {
    "ecoli": Path("/media/marcelo_baez/HD_Disc1/.S2F/output/83333/83333_PLM/query_embeddings"),
    "human": Path("/media/marcelo_baez/HD_Disc1/.S2F/data/PLM/embeddings/human_9606_all_swissprot"),
}
TAXON_BY_ORGANISM = {
    "ecoli": "83333",
    "human": "9606",
}
S2F_SELECTED_SCORES_PATH = DATA_DIR / "s2f_selected_scores_ecoli.csv"
COMPETITORS_DIR = PROJECT_ROOT / "competitors"
RANDOM_SEED = 42
K_VALUES = [1, 3, 5, 10]
IDENTITY_FILTERS = [
    {"key": "all", "label": "All neighbors", "max_identity": None},
    {"key": "lt50", "label": "<50% sequence identity", "max_identity": 50.0},
    {"key": "lt40", "label": "<40% sequence identity", "max_identity": 40.0},
    {"key": "lt30", "label": "<30% sequence identity", "max_identity": 30.0},
]
METRIC_COLUMNS = [
    "overall::AUC", "overall::AUPR", "overall::F_max", "overall::smin",
    "AUC per-gene", "AUPR per-gene", "F_max per-gene", "smin per-gene",
    "AUC per-term", "AUPR per-term", "F_max per-term", "smin per-term",
]

print("Project root:", PROJECT_ROOT)
print("Frontend data:", DATA_DIR)
print("Selected-term Smin IC:", "estimated from the selected 60-by-3 gold matrix")

Project root: /home/marcelo_baez/paccanaro-lab/S2F
Frontend data: /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data
Selected-term Smin IC: estimated from the selected 60-by-3 gold matrix


In [2]:
def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


index_payload = load_json(INDEX_PATH)
probe_payload = load_json(PROBE_RESULTS_PATH)

datasets = {}
for descriptor in index_payload["datasets"]:
    dataset = load_json(PROJECT_ROOT / "notebooks" / "esm_go_explorer" / descriptor["path"])
    datasets[descriptor["dataset_id"]] = dataset

caches = {}
cache_indices = {}
for organism_key, cache_path in CACHE_BY_ORGANISM.items():
    cache = plm.load_embedding_cache(cache_path, validate=False)
    if cache is None:
        raise RuntimeError(f"Could not load embedding cache for {organism_key}: {cache_path}")
    caches[organism_key] = cache
    cache_indices[organism_key] = {protein_id: idx for idx, protein_id in enumerate(cache.ids)}
    print(organism_key, cache.embeddings.shape, cache_path)

print(f"Loaded {len(datasets)} explorer datasets:", ", ".join(datasets))

ecoli (16075, 1280) /media/marcelo_baez/HD_Disc1/.S2F/output/83333/83333_PLM/query_embeddings
human (20420, 1280) /media/marcelo_baez/HD_Disc1/.S2F/data/PLM/embeddings/human_9606_all_swissprot
Loaded 6 explorer datasets: ecoli_mf, ecoli_bp, ecoli_cc, human_mf, human_bp, human_cc


## Metric Helpers

The selected-term probe is evaluated as a PFP ranking problem over three GO terms. Each method produces scores for the same selected proteins and selected GO terms; unscored pairs remain zero. `HX_py` then computes the same metric families used in the competitor notebook. In this constrained view, IC is computed from the selected-term gold matrix so `Smin` is comparable across probe methods within the same dataset.

In [3]:
def dataframe_records(df):
    return json.loads(df.replace({np.nan: None}).to_json(orient="records"))


def finite_or_none(value):
    if value is None:
        return None
    try:
        value = float(value)
    except Exception:
        return value
    return value if math.isfinite(value) else None


def information_content_for_dataset(dataset):
    gold = selected_gold_matrix(dataset)
    term_frequency = gold.sum(axis=0) / max(gold.shape[0], 1)
    term_frequency = np.clip(term_frequency, np.finfo(float).tiny, 1.0)
    return -np.log(term_frequency).astype(np.float32)


def flatten_metrics(nested_metrics):
    flat = {}
    for block, metrics in nested_metrics.items():
        for key, value in metrics.items():
            if isinstance(value, (list, tuple, dict)):
                continue
            arr = np.asarray(value)
            if arr.ndim == 0:
                label = key if block != "overall" else f"{block}::{key}"
                flat[label] = finite_or_none(arr)
    return flat


def selected_gold_matrix(dataset):
    proteins = dataset["proteins"]
    groups = dataset["groups"]
    term_to_idx = {group["go_id"]: int(group["group_index"]) for group in groups}
    gold = np.zeros((len(proteins), len(groups)), dtype=np.float32)
    for row, protein in enumerate(proteins):
        gold[row, term_to_idx[protein["go_id"]]] = 1.0
    return gold


def score_rows_to_matrix(dataset, score_rows):
    proteins = dataset["proteins"]
    groups = dataset["groups"]
    protein_to_idx = {protein["protein_id"]: idx for idx, protein in enumerate(proteins)}
    term_to_idx = {group["go_id"]: idx for idx, group in enumerate(groups)}
    prediction = np.zeros((len(proteins), len(groups)), dtype=np.float32)
    if not score_rows.empty:
        rows = score_rows[score_rows["protein_id"].isin(protein_to_idx) & score_rows["term_id"].isin(term_to_idx)].copy()
        if not rows.empty:
            rows["score"] = pd.to_numeric(rows["score"], errors="coerce").fillna(0.0)
            rows = rows.groupby(["protein_id", "term_id"], as_index=False)["score"].max()
            for row in rows.itertuples(index=False):
                prediction[protein_to_idx[row.protein_id], term_to_idx[row.term_id]] = float(row.score)
    return prediction


def evaluate_selected_term_scores(dataset_id, dataset, method, method_key, method_family, score_rows, extra=None):
    score_rows = score_rows.copy()
    score_rows["score"] = pd.to_numeric(score_rows["score"], errors="coerce")
    score_rows = score_rows.dropna(subset=["protein_id", "term_id", "score"])
    prediction = score_rows_to_matrix(dataset, score_rows)
    gold = selected_gold_matrix(dataset)
    ic = information_content_for_dataset(dataset)
    measure = HX_py(prediction, ic, organism_id=f"{method_key}_{dataset_id}", verbose=False)
    results = {
        "overall": measure.compute_overall(gold),
        "per_gene": measure.compute_per_gene(gold),
        "per_term": measure.compute_per_term(gold),
    }
    proteins = {protein["protein_id"] for protein in dataset["proteins"]}
    terms = {group["go_id"] for group in dataset["groups"]}
    filtered_scores = score_rows[score_rows["protein_id"].isin(proteins) & score_rows["term_id"].isin(terms)]
    row = {
        "dataset_id": dataset_id,
        "organism": dataset["metadata"]["organism"],
        "organism_key": dataset["metadata"]["organism_key"],
        "taxon": dataset["metadata"]["taxon"],
        "domain": dataset["metadata"]["domain"],
        "domain_name": dataset["metadata"]["domain_name"],
        "method": method,
        "method_key": method_key,
        "method_family": method_family,
        "scope": "selected_go_terms",
        "proteins_considered": int(gold.shape[0]),
        "terms_considered": int(gold.shape[1]),
        "total_annotations": int(gold.sum()),
        "matrix_shape": str(tuple(gold.shape)),
        "score_pairs": int(len(filtered_scores)),
        "proteins_with_scores": int(filtered_scores["protein_id"].nunique()) if not filtered_scores.empty else 0,
        "terms_with_scores": int(filtered_scores["term_id"].nunique()) if not filtered_scores.empty else 0,
    }
    row.update(flatten_metrics(results))
    if extra:
        row.update(extra)
    return row


print("Metric helper functions ready.")

Metric helper functions ready.


## Scored Predictions From ESM Probes

For kNN, the score for each GO term is the share of positive cosine-similarity weight assigned to that group among the eligible neighbors. For the linear probe, the score is the leave-one-out logistic-regression probability for each selected GO term.

In [4]:
def dataset_embeddings(dataset):
    organism_key = dataset["metadata"]["organism_key"]
    cache = caches[organism_key]
    index = cache_indices[organism_key]
    positions = [index[protein["protein_id"]] for protein in dataset["proteins"]]
    return np.asarray(cache.embeddings[positions], dtype=np.float32)


def all_selected_pairs(dataset, default_score=0.0):
    return pd.DataFrame(
        [
            {"protein_id": protein["protein_id"], "term_id": group["go_id"], "score": float(default_score)}
            for protein in dataset["proteins"]
            for group in dataset["groups"]
        ]
    )


def knn_score_rows(dataset_id, dataset, k, identity_filter):
    proteins = dataset["proteins"]
    groups = dataset["groups"]
    labels = np.asarray([protein["group_index"] for protein in proteins], dtype=int)
    cosine_similarity = np.asarray(dataset["matrices"]["cosine_similarity"]["matrix"], dtype=np.float32)
    identity = np.asarray(probe_payload["knn"]["datasets"][dataset_id]["sequence_identity_matrix"], dtype=np.float32)
    max_identity = identity_filter["max_identity"]
    rows = []
    for query_index, protein in enumerate(proteins):
        candidates = [idx for idx in range(len(proteins)) if idx != query_index]
        if max_identity is not None:
            candidates = [idx for idx in candidates if identity[query_index, idx] < max_identity]
        ranked = sorted(candidates, key=lambda idx: (-cosine_similarity[query_index, idx], idx))
        eligible = ranked[:k]
        votes = defaultdict(float)
        total = 0.0
        for neighbor_index in eligible:
            weight = max(float(cosine_similarity[query_index, neighbor_index]), 0.0)
            votes[int(labels[neighbor_index])] += weight
            total += weight
        if total <= 0.0 and eligible:
            for neighbor_index in eligible:
                votes[int(labels[neighbor_index])] += 1.0
            total = float(len(eligible))
        for group in groups:
            group_index = int(group["group_index"])
            rows.append({
                "dataset_id": dataset_id,
                "protein_id": protein["protein_id"],
                "term_id": group["go_id"],
                "score": votes.get(group_index, 0.0) / total if total > 0.0 else 0.0,
                "method": f"ESM kNN k={k}" + ("" if identity_filter["key"] == "all" else f" {identity_filter['label']}"),
                "method_key": f"knn_{identity_filter['key']}_k{k}",
                "method_family": "ESM kNN",
                "identity_filter": identity_filter["key"],
                "identity_filter_label": identity_filter["label"],
                "k": k,
            })
    return pd.DataFrame(rows)


def linear_probe_score_rows(dataset_id, dataset):
    proteins = dataset["proteins"]
    groups = dataset["groups"]
    labels = np.asarray([protein["group_index"] for protein in proteins], dtype=int)
    X = dataset_embeddings(dataset)
    rows = []
    for test_index, protein in enumerate(proteins):
        train_mask = np.ones(len(proteins), dtype=bool)
        train_mask[test_index] = False
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=1.0,
                solver="lbfgs",
                class_weight="balanced",
                max_iter=5000,
                random_state=RANDOM_SEED,
            ),
        )
        model.fit(X[train_mask], labels[train_mask])
        probabilities = model.predict_proba(X[[test_index]])[0]
        class_to_probability = {int(cls): float(prob) for cls, prob in zip(model.classes_, probabilities)}
        for group in groups:
            group_index = int(group["group_index"])
            rows.append({
                "dataset_id": dataset_id,
                "protein_id": protein["protein_id"],
                "term_id": group["go_id"],
                "score": class_to_probability.get(group_index, 0.0),
                "method": "Linear probe LOOCV",
                "method_key": "linear_probe",
                "method_family": "Linear probe",
            })
    return pd.DataFrame(rows)


esm_score_tables = {}
for dataset_id, dataset in datasets.items():
    tables = []
    for identity_filter in IDENTITY_FILTERS:
        for k in K_VALUES:
            tables.append(knn_score_rows(dataset_id, dataset, k, identity_filter))
    tables.append(linear_probe_score_rows(dataset_id, dataset))
    esm_score_tables[dataset_id] = pd.concat(tables, ignore_index=True)
    print(dataset_id, len(esm_score_tables[dataset_id]), "ESM probe score rows")

print("ESM probe score tables built.")

ecoli_mf 3060 ESM probe score rows
ecoli_bp 3060 ESM probe score rows
ecoli_cc 3060 ESM probe score rows
human_mf 3060 ESM probe score rows
human_bp 3060 ESM probe score rows
human_cc 3060 ESM probe score rows
ESM probe score tables built.


## Selected-Term S2F and Competitor Scores

These loaders extract only the selected proteins and selected GO terms. Missing pairs remain zero in the metric matrix. Competitor scores are local-context checks for the E. coli selected proteins; the full fair competitor benchmark remains separate.

In [5]:
PREDICTION_COLUMNS = ["protein_id", "term_id", "score"]


def normalise_prediction_frame(df, model, organism):
    if df is None or df.empty:
        return pd.DataFrame(columns=[*PREDICTION_COLUMNS, "model", "organism"])
    result = df.rename(columns={"Protein": "protein_id", "GO ID": "term_id", "Score": "score"}).copy()
    missing = [column for column in PREDICTION_COLUMNS if column not in result.columns]
    if missing:
        raise ValueError(f"{model} predictions for {organism} are missing columns: {missing}")
    result = result[PREDICTION_COLUMNS].copy()
    result["protein_id"] = result["protein_id"].astype(str).str.strip()
    result["term_id"] = result["term_id"].astype(str).str.strip()
    result["score"] = pd.to_numeric(result["score"], errors="coerce")
    result = result.dropna(subset=["protein_id", "term_id", "score"])
    result = result[result["protein_id"].ne("") & result["term_id"].str.match(r"^GO:\d+$")]
    result["model"] = model
    result["organism"] = str(organism)
    if result.empty:
        return pd.DataFrame(columns=[*PREDICTION_COLUMNS, "model", "organism"])
    result = result.groupby(["protein_id", "term_id", "model", "organism"], as_index=False, sort=False)["score"].max()
    return result


def selected_s2f_scores(dataset_id, dataset):
    if dataset["metadata"]["organism_key"] != "ecoli" or not S2F_SELECTED_SCORES_PATH.exists():
        return pd.DataFrame(columns=PREDICTION_COLUMNS)
    scores = pd.read_csv(S2F_SELECTED_SCORES_PATH).rename(columns={"go_id": "term_id", "s2f_score": "score"})
    proteins = {protein["protein_id"] for protein in dataset["proteins"]}
    terms = {group["go_id"] for group in dataset["groups"]}
    scores = scores[scores["protein_id"].isin(proteins) & scores["term_id"].isin(terms)].copy()
    return scores[PREDICTION_COLUMNS]


TALE_ONTOLOGY_FILES = {
    "bp": COMPETITORS_DIR / "TALE" / "bp" / "output.txt",
    "mf": COMPETITORS_DIR / "TALE" / "mf" / "output.txt",
    "cc": COMPETITORS_DIR / "TALE" / "cc" / "output.txt",
}
TALE_PATTERN = re.compile(
    r"^(?P<protein>\S+)\s+\('(?P<term>GO:\d+)',.*\)\s+(?P<score>[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)\s*$"
)


def selected_tale_scores(dataset):
    domain_key = dataset["metadata"]["domain"].lower()
    path = TALE_ONTOLOGY_FILES.get(domain_key)
    proteins = {protein["protein_id"] for protein in dataset["proteins"]}
    terms = {group["go_id"] for group in dataset["groups"]}
    rows = []
    if path is None or not path.exists():
        return pd.DataFrame(columns=PREDICTION_COLUMNS)
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            match = TALE_PATTERN.match(line.rstrip())
            if match is None:
                continue
            protein = match.group("protein")
            term = match.group("term")
            if protein in proteins and term in terms:
                rows.append((protein, term, match.group("score")))
    return pd.DataFrame(rows, columns=PREDICTION_COLUMNS)


ATGO_ONTOLOGY_DIRS = {
    "BP": COMPETITORS_DIR / "ATGO" / "BP" / "result",
    "MF": COMPETITORS_DIR / "ATGO" / "MF" / "result",
    "CC": COMPETITORS_DIR / "ATGO" / "CC" / "result",
}


def _atgo_prediction_file(protein, ontology):
    result_dir = ATGO_ONTOLOGY_DIRS[ontology] / protein
    preferred = result_dir / f"final_combine_{ontology}_new"
    fallback = result_dir / f"final_combine_{ontology}"
    if preferred.exists():
        return preferred
    if fallback.exists():
        return fallback
    return None


def selected_atgo_scores(dataset):
    ontology = dataset["metadata"]["domain"]
    proteins = {protein["protein_id"] for protein in dataset["proteins"]}
    terms = {group["go_id"] for group in dataset["groups"]}
    rows = []
    if ontology not in ATGO_ONTOLOGY_DIRS or not ATGO_ONTOLOGY_DIRS[ontology].exists():
        return pd.DataFrame(columns=PREDICTION_COLUMNS)
    for protein in sorted(proteins):
        path = _atgo_prediction_file(protein, ontology)
        if path is None:
            continue
        with path.open("r", encoding="utf-8", errors="replace") as handle:
            for line in handle:
                parts = line.strip().split()
                if len(parts) < 3 or parts[0] not in terms:
                    continue
                rows.append((protein, parts[0], parts[-1]))
    return pd.DataFrame(rows, columns=PREDICTION_COLUMNS)


PANDA2_BY_DATASET_DIR = COMPETITORS_DIR / "PANDA2" / "by_dataset"
PANDA2_HEADER_TOKENS = {"AUTHOR", "MODEL", "KEYWORDS", "END"}


def selected_panda2_scores(dataset):
    if dataset["metadata"]["organism_key"] != "ecoli":
        return pd.DataFrame(columns=PREDICTION_COLUMNS)
    path = PANDA2_BY_DATASET_DIR / "83333" / "83333.2" / "panda2_prediction.txt"
    if not path.exists():
        return pd.DataFrame(columns=PREDICTION_COLUMNS)
    proteins = {protein["protein_id"] for protein in dataset["proteins"]}
    terms = {group["go_id"] for group in dataset["groups"]}
    rows = []
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            parts = line.strip().split()
            if not parts or parts[0] in PANDA2_HEADER_TOKENS:
                continue
            if len(parts) != 3:
                continue
            protein, term, score = parts
            if protein in proteins and term in terms:
                rows.append((protein, term, score))
    return pd.DataFrame(rows, columns=PREDICTION_COLUMNS)


selected_competitor_loaders = {
    "TALE": ("TALE", "competitor", selected_tale_scores),
    "ATGO": ("ATGO", "competitor", selected_atgo_scores),
    "PANDA2": ("PANDA2", "competitor", selected_panda2_scores),
}

print("Selected-term S2F and competitor loaders ready.")

Selected-term S2F and competitor loaders ready.


## Evaluate Selected-Term PFP Metrics

In [6]:
metric_rows = []
pair_score_rows = []

for dataset_id, dataset in datasets.items():
    esm_rows = esm_score_tables[dataset_id]
    for method_key, score_rows in esm_rows.groupby("method_key", sort=False):
        method = str(score_rows["method"].iloc[0])
        method_family = str(score_rows["method_family"].iloc[0])
        extra = {
            "display_in_frontend": bool(method_key in {"knn_all_k5", "knn_lt30_k5", "linear_probe"}),
            "k": finite_or_none(score_rows["k"].iloc[0]) if "k" in score_rows and score_rows["k"].notna().any() else None,
            "identity_filter": score_rows["identity_filter"].iloc[0] if "identity_filter" in score_rows and score_rows["identity_filter"].notna().any() else None,
            "identity_filter_label": score_rows["identity_filter_label"].iloc[0] if "identity_filter_label" in score_rows and score_rows["identity_filter_label"].notna().any() else None,
        }
        score_df = score_rows[["protein_id", "term_id", "score"]].copy()
        metric_rows.append(evaluate_selected_term_scores(dataset_id, dataset, method, method_key, method_family, score_df, extra=extra))
        exported = score_df.copy()
        exported["dataset_id"] = dataset_id
        exported["method"] = method
        exported["method_key"] = method_key
        exported["method_family"] = method_family
        pair_score_rows.append(exported)

    s2f_scores = selected_s2f_scores(dataset_id, dataset)
    if not s2f_scores.empty:
        metric_rows.append(evaluate_selected_term_scores(
            dataset_id,
            dataset,
            "S2F selected-term score",
            "s2f_selected",
            "S2F",
            s2f_scores,
            extra={"display_in_frontend": True},
        ))
        exported = s2f_scores.copy()
        exported["dataset_id"] = dataset_id
        exported["method"] = "S2F selected-term score"
        exported["method_key"] = "s2f_selected"
        exported["method_family"] = "S2F"
        pair_score_rows.append(exported)

    if dataset["metadata"]["organism_key"] == "ecoli":
        for method_key, (method, method_family, loader) in selected_competitor_loaders.items():
            scores = loader(dataset)
            if scores.empty:
                # Keep a frontend-visible zero-score row so missing selected-term output is explicit.
                scores = all_selected_pairs(dataset, default_score=0.0)
                availability_note = "No selected score pairs found locally; all selected pairs evaluated as zero."
            else:
                availability_note = "Selected score pairs found locally."
            metric_rows.append(evaluate_selected_term_scores(
                dataset_id,
                dataset,
                method,
                method_key.lower(),
                method_family,
                scores,
                extra={"display_in_frontend": True, "availability_note": availability_note},
            ))
            exported = scores.copy()
            exported["dataset_id"] = dataset_id
            exported["method"] = method
            exported["method_key"] = method_key.lower()
            exported["method_family"] = method_family
            pair_score_rows.append(exported)

PFP_METRICS_DF = pd.DataFrame(metric_rows)
PFP_PAIR_SCORES_DF = pd.concat(pair_score_rows, ignore_index=True)

sort_columns = ["dataset_id", "method_family", "method_key"]
PFP_METRICS_DF = PFP_METRICS_DF.sort_values(sort_columns).reset_index(drop=True)
print("Selected-term PFP metrics:", PFP_METRICS_DF.shape)
print("Pair scores:", PFP_PAIR_SCORES_DF.shape)
print(PFP_METRICS_DF[PFP_METRICS_DF["display_in_frontend"].fillna(False)][[
    "dataset_id", "method", "score_pairs", "overall::AUC", "overall::AUPR", "overall::F_max", "overall::smin"
]].to_string(index=False))

Selected-term PFP metrics: (114, 43)
Pair scores: (20520, 7)
dataset_id                             method  score_pairs  overall::AUC  overall::AUPR  overall::F_max  overall::smin
  ecoli_bp                        ESM kNN k=5          180      0.849722       0.592588        0.745763       0.389280
  ecoli_bp ESM kNN k=5 <30% sequence identity          180      0.845972       0.586939        0.745763       0.389280
  ecoli_bp                 Linear probe LOOCV          180      0.899861       0.828398        0.823529       0.272201
  ecoli_bp            S2F selected-term score          180      0.705833       0.667853        0.632653       0.544402
  ecoli_bp                               ATGO          180      0.500000       0.333333        0.000000       2.197225
  ecoli_bp                             PANDA2          180      0.500000       0.333333        0.000000       2.197225
  ecoli_bp                               TALE          180      0.500000       0.333333        0.000000   

## Full Benchmark Competitor Context

This context is loaded from the existing competitor evaluation CSV files. It is not mixed with the selected 60-protein probe metrics because the evaluation universe is different.

In [7]:
def load_competitor_context():
    frames = []
    for path in sorted((PROJECT_ROOT / "figures").glob("*/old/competitors_comparison.csv")):
        df = pd.read_csv(path)
        df["source_csv"] = str(path.relative_to(PROJECT_ROOT))
        frames.append(df)
    if frames:
        result = pd.concat(frames, ignore_index=True)
    else:
        result = pd.DataFrame()
    return result


COMPETITOR_CONTEXT_DF = load_competitor_context()
if COMPETITOR_CONTEXT_DF.empty:
    print("No full benchmark competitor context CSVs found.")
else:
    print("Full benchmark competitor context:", COMPETITOR_CONTEXT_DF.shape)
    print(COMPETITOR_CONTEXT_DF[[
        "organism", "model", "proteins_considered", "terms_considered",
        "overall::AUC", "overall::AUPR", "overall::F_max", "overall::smin"
    ]].to_string(index=False))

Full benchmark competitor context: (12, 33)
 organism  model  proteins_considered  terms_considered  overall::AUC  overall::AUPR  overall::F_max  overall::smin
  1111708    S2F                   33                93      0.831826       0.301715        0.412478      60.930628
  1111708   TALE                   33                93      0.741410       0.219811        0.375921      57.698492
  1111708   ATGO                   33                93      0.626935       0.174666        0.280475      61.306422
  1111708 PANDA2                   33                93      0.712267       0.233735        0.345238      61.064334
   223283    S2F                    3                 4      0.592593       0.710446        0.900000       2.918296
   223283   TALE                    3                 4      0.944444       0.650393        0.941176       1.723308
   223283   ATGO                    3                 4      0.500000       0.750000        0.000000       4.641604
   223283 PANDA2            

## Export Frontend Data

In [8]:
def dataset_method_records(metrics_df, dataset_id):
    subset = metrics_df[metrics_df["dataset_id"] == dataset_id].copy()
    return dataframe_records(subset)


selected_datasets = {}
for dataset_id, dataset in datasets.items():
    selected_datasets[dataset_id] = {
        "metadata": dataset["metadata"],
        "groups": dataset["groups"],
        "methods": dataset_method_records(PFP_METRICS_DF, dataset_id),
    }

payload = {
    "schema_version": 1,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": {
        "metric_implementation": "Measures.measures.HX_py",
        "selected_term_smin_ic": "Information content is estimated from the selected 60-by-3 gold matrix; full benchmark Smin uses the original GOA/ontology competitor notebook values.",
        "selected_term_scope": "60 proteins by 3 selected direct GO groups per dataset",
        "full_benchmark_source": "figures/*/old/competitors_comparison.csv",
        "datasets": list(datasets.keys()),
    },
    "metric_columns": METRIC_COLUMNS,
    "selected_term_metrics": {
        "description": "PFP metrics for scored selected GO-term predictions over the same proteins used in the ESM explorer.",
        "rows": dataframe_records(PFP_METRICS_DF),
        "datasets": selected_datasets,
    },
    "benchmark_context": {
        "description": "Full competitor benchmark metrics from competitors_performance.ipynb outputs. This context uses a different protein and term universe from the selected ESM probe.",
        "rows": dataframe_records(COMPETITOR_CONTEXT_DF) if not COMPETITOR_CONTEXT_DF.empty else [],
    },
}

with PFP_JSON_PATH.open("w", encoding="utf-8") as handle:
    json.dump(payload, handle, separators=(",", ":"), allow_nan=False)
PFP_METRICS_DF.to_csv(PFP_METRICS_CSV_PATH, index=False)
PFP_PAIR_SCORES_DF.to_csv(PFP_PAIR_SCORES_CSV_PATH, index=False)
COMPETITOR_CONTEXT_DF.to_csv(PFP_COMPETITOR_CONTEXT_CSV_PATH, index=False)

index_payload["pfp_metrics_path"] = "data/probe_pfp_metrics.json"
index_payload["pfp_metrics_schema_version"] = 1
with INDEX_PATH.open("w", encoding="utf-8") as handle:
    json.dump(index_payload, handle, indent=2, sort_keys=True, allow_nan=False)

print("Wrote", PFP_JSON_PATH)
print("Wrote", PFP_METRICS_CSV_PATH)
print("Wrote", PFP_PAIR_SCORES_CSV_PATH)
print("Wrote", PFP_COMPETITOR_CONTEXT_CSV_PATH)
print("Updated", INDEX_PATH)

Wrote /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data/probe_pfp_metrics.json
Wrote /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data/probe_pfp_metrics.csv
Wrote /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data/probe_pfp_pair_scores.csv
Wrote /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data/competitor_context_metrics.csv
Updated /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data/index.json


## Spreadsheet Export

This cell prepares the spreadsheet-style provenance view used by the frontend. Term rows summarize each method and selected GO annotation, protein-term rows show the score behind each selected annotation, and benchmark rows keep the full competitor context as aggregate-only evidence.


In [16]:
def finite_metric(value):
    return finite_or_none(value)


def csv_or_memory(path, fallback_df):
    if path.exists():
        return pd.read_csv(path)
    return fallback_df.copy()


def method_metadata_table(metrics_df):
    metadata = {}
    for row in metrics_df.replace({np.nan: None}).to_dict(orient="records"):
        metadata[(row.get("dataset_id"), row.get("method_key"))] = row
    return metadata


def dataset_lookup_tables(dataset):
    groups = {group["go_id"]: group for group in dataset["groups"]}
    proteins = {protein["protein_id"]: protein for protein in dataset["proteins"]}
    return groups, proteins


def score_vector_for_term(dataset, score_rows, term_id):
    proteins = dataset["proteins"]
    score_map = {}
    if not score_rows.empty:
        rows = score_rows[score_rows["term_id"] == term_id].copy()
        if not rows.empty:
            rows["score"] = pd.to_numeric(rows["score"], errors="coerce").fillna(0.0)
            rows = rows.groupby("protein_id", as_index=False)["score"].max()
            score_map = dict(zip(rows["protein_id"], rows["score"]))
    return np.asarray([float(score_map.get(protein["protein_id"], 0.0)) for protein in proteins], dtype=np.float32)


def term_metric_values(dataset, score_rows, term_id):
    term_to_idx = {group["go_id"]: int(group["group_index"]) for group in dataset["groups"]}
    term_idx = term_to_idx[term_id]
    gold = selected_gold_matrix(dataset)[:, [term_idx]]
    prediction = score_vector_for_term(dataset, score_rows, term_id).reshape(-1, 1)
    ic = information_content_for_dataset(dataset)[[term_idx]]
    measure = HX_py(prediction, ic, organism_id=f"spreadsheet_{dataset['metadata']['dataset_id']}_{term_id}", verbose=False)
    return measure.compute_overall(gold)


def build_term_rows(metrics_df, pair_scores_df):
    method_meta = method_metadata_table(metrics_df)
    rows = []
    for dataset_id, dataset in datasets.items():
        metadata = dataset["metadata"]
        groups, _ = dataset_lookup_tables(dataset)
        dataset_scores = pair_scores_df[pair_scores_df["dataset_id"] == dataset_id].copy()
        if dataset_scores.empty:
            continue
        for (method_key, term_id), score_rows in dataset_scores.groupby(["method_key", "term_id"], sort=False):
            group = groups.get(term_id)
            if group is None:
                continue
            meta = method_meta.get((dataset_id, method_key), {})
            metrics = term_metric_values(dataset, dataset_scores[dataset_scores["method_key"] == method_key], term_id)
            scores = pd.to_numeric(score_rows["score"], errors="coerce").dropna()
            positives = sum(1 for protein in dataset["proteins"] if protein["go_id"] == term_id)
            rows.append({
                "row_type": "term",
                "dataset_id": dataset_id,
                "organism": metadata["organism"],
                "organism_key": metadata["organism_key"],
                "taxon": metadata["taxon"],
                "domain": metadata["domain"],
                "domain_name": metadata["domain_name"],
                "go_id": term_id,
                "go_name": group.get("name"),
                "group_index": group.get("group_index"),
                "method": score_rows["method"].iloc[0],
                "method_key": method_key,
                "method_family": score_rows["method_family"].iloc[0],
                "display_in_frontend": bool(meta.get("display_in_frontend", False)),
                "drilldown_available": True,
                "proteins_considered": len(dataset["proteins"]),
                "positives": positives,
                "negatives": len(dataset["proteins"]) - positives,
                "score_pairs": int(len(score_rows)),
                "score_mean": finite_metric(scores.mean() if not scores.empty else None),
                "score_min": finite_metric(scores.min() if not scores.empty else None),
                "score_max": finite_metric(scores.max() if not scores.empty else None),
                "AUC": finite_metric(metrics.get("AUC")),
                "AUPR": finite_metric(metrics.get("AUPR")),
                "F_max": finite_metric(metrics.get("F_max")),
                "smin": finite_metric(metrics.get("smin")),
                "source_type": "selected_go_terms",
                "source_file": "probe_pfp_pair_scores.csv",
                "source_note": meta.get("availability_note") or "Scores evaluated over the selected 60 proteins and three direct GO terms.",
            })
    return pd.DataFrame(rows)


def build_pair_rows(metrics_df, pair_scores_df):
    method_meta = method_metadata_table(metrics_df)
    rows = []
    for dataset_id, dataset in datasets.items():
        metadata = dataset["metadata"]
        groups, proteins = dataset_lookup_tables(dataset)
        dataset_scores = pair_scores_df[pair_scores_df["dataset_id"] == dataset_id].copy()
        for row in dataset_scores.itertuples(index=False):
            protein = proteins.get(row.protein_id)
            group = groups.get(row.term_id)
            if protein is None or group is None:
                continue
            meta = method_meta.get((dataset_id, row.method_key), {})
            rows.append({
                "row_type": "pair",
                "dataset_id": dataset_id,
                "organism": metadata["organism"],
                "organism_key": metadata["organism_key"],
                "taxon": metadata["taxon"],
                "domain": metadata["domain"],
                "domain_name": metadata["domain_name"],
                "protein_id": row.protein_id,
                "protein_direct_go_id": protein.get("go_id"),
                "protein_direct_go_name": protein.get("go_name"),
                "go_id": row.term_id,
                "go_name": group.get("name"),
                "group_index": group.get("group_index"),
                "true_label": int(protein.get("go_id") == row.term_id),
                "score": finite_metric(row.score),
                "method": row.method,
                "method_key": row.method_key,
                "method_family": row.method_family,
                "display_in_frontend": bool(meta.get("display_in_frontend", False)),
                "drilldown_available": False,
                "source_type": "selected_go_terms",
                "source_file": "probe_pfp_pair_scores.csv",
                "source_note": meta.get("availability_note") or "Protein-term score used to compute selected-term metrics.",
            })
    return pd.DataFrame(rows)


def build_benchmark_rows(context_df):
    rows = []
    if context_df.empty:
        return pd.DataFrame(rows)
    for row in context_df.replace({np.nan: None}).to_dict(orient="records"):
        rows.append({
            "row_type": "benchmark",
            "organism": row.get("organism"),
            "taxon": row.get("organism"),
            "method": row.get("model"),
            "method_key": str(row.get("model", "")).lower(),
            "method_family": row.get("model"),
            "display_in_frontend": True,
            "drilldown_available": False,
            "proteins_considered": finite_metric(row.get("proteins_considered")),
            "terms_considered": finite_metric(row.get("terms_considered")),
            "total_annotations": finite_metric(row.get("total_annotations")),
            "matrix_shape": row.get("matrix_shape"),
            "AUC": finite_metric(row.get("overall::AUC")),
            "AUPR": finite_metric(row.get("overall::AUPR")),
            "F_max": finite_metric(row.get("overall::F_max")),
            "smin": finite_metric(row.get("overall::smin")),
            "source_type": "full_benchmark_context",
            "source_file": row.get("source_csv"),
            "source_note": "Full benchmark aggregate from competitors_performance.ipynb; per-annotation drill-down is not available in this artifact.",
        })
    return pd.DataFrame(rows)


metrics_for_spreadsheet = csv_or_memory(PFP_METRICS_CSV_PATH, PFP_METRICS_DF)
pair_scores_for_spreadsheet = csv_or_memory(PFP_PAIR_SCORES_CSV_PATH, PFP_PAIR_SCORES_DF)
context_for_spreadsheet = csv_or_memory(PFP_COMPETITOR_CONTEXT_CSV_PATH, COMPETITOR_CONTEXT_DF)

SPREADSHEET_TERM_ROWS_DF = build_term_rows(metrics_for_spreadsheet, pair_scores_for_spreadsheet)
SPREADSHEET_PAIR_ROWS_DF = build_pair_rows(metrics_for_spreadsheet, pair_scores_for_spreadsheet)
SPREADSHEET_BENCHMARK_ROWS_DF = build_benchmark_rows(context_for_spreadsheet)
SPREADSHEET_ALL_ROWS_DF = pd.concat(
    [SPREADSHEET_TERM_ROWS_DF, SPREADSHEET_PAIR_ROWS_DF, SPREADSHEET_BENCHMARK_ROWS_DF],
    ignore_index=True,
    sort=False,
)

spreadsheet_payload = {
    "schema_version": 1,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": {
        "metric_implementation": "Measures.measures.HX_py",
        "selected_term_scope": "60 proteins by 3 selected direct GO groups per dataset",
        "pair_score_source": "probe_pfp_pair_scores.csv",
        "benchmark_source": "competitor_context_metrics.csv",
        "drilldown_policy": "Selected-term rows can drill down to protein-term scores; benchmark rows are aggregate-only.",
    },
    "term_rows": dataframe_records(SPREADSHEET_TERM_ROWS_DF),
    "pair_rows": dataframe_records(SPREADSHEET_PAIR_ROWS_DF),
    "benchmark_rows": dataframe_records(SPREADSHEET_BENCHMARK_ROWS_DF),
}

with PFP_SPREADSHEET_JSON_PATH.open("w", encoding="utf-8") as handle:
    json.dump(spreadsheet_payload, handle, separators=(",", ":"), allow_nan=False)
SPREADSHEET_ALL_ROWS_DF.to_csv(PFP_SPREADSHEET_CSV_PATH, index=False)

index_payload = load_json(INDEX_PATH)
index_payload["pfp_spreadsheet_path"] = "data/pfp_spreadsheet.json"
index_payload["pfp_spreadsheet_schema_version"] = 1
with INDEX_PATH.open("w", encoding="utf-8") as handle:
    json.dump(index_payload, handle, indent=2, sort_keys=True, allow_nan=False)

print("Spreadsheet term rows:", SPREADSHEET_TERM_ROWS_DF.shape)
print("Spreadsheet pair rows:", SPREADSHEET_PAIR_ROWS_DF.shape)
print("Spreadsheet benchmark rows:", SPREADSHEET_BENCHMARK_ROWS_DF.shape)
print("Wrote", PFP_SPREADSHEET_JSON_PATH)
print("Wrote", PFP_SPREADSHEET_CSV_PATH)
print("Updated", INDEX_PATH)


Spreadsheet term rows: (342, 29)
Spreadsheet pair rows: (20520, 23)
Spreadsheet benchmark rows: (12, 19)
Wrote /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data/pfp_spreadsheet.json
Wrote /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data/pfp_spreadsheet.csv
Updated /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data/index.json


## Interpretation Notes

- The selected-term PFP metrics ask whether the three displayed GO labels are recoverable as scored predictions from frozen ESM embeddings.
- Selected-term `Smin` uses information content estimated inside the balanced 60-by-3 selected matrix; the full benchmark context uses the original GOA/ontology `Smin` values.
- Strong ESM kNN performance means the labels are visible in local embedding geometry.
- Strong linear-probe performance with weaker kNN means the information is present but needs a simple decoder.
- Competitor selected-term rows are local overlap checks, not the fair full benchmark.
- The full benchmark section should be used as context for real PFP performance because it evaluates many more GO terms and a different protein universe.